# 06 - Model Evaluation & Comparison

This notebook evaluates all trained models and creates comparison visualizations.

## Cell 1: Setup & Load All Models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
PROJECT_DIR = '/content/drive/MyDrive/smart-ambulance-ml'
sys.path.append(PROJECT_DIR)

!pip install -q xgboost lightgbm joblib matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load all processed data
severity_data = np.load(f'{PROJECT_DIR}/datasets/processed/severity_processed.npz', allow_pickle=True)
eta_data = np.load(f'{PROJECT_DIR}/datasets/processed/eta_processed.npz', allow_pickle=True)
bed_data = np.load(f'{PROJECT_DIR}/datasets/processed/bed_processed.npz', allow_pickle=True)

# Load model registry
with open(f'{PROJECT_DIR}/models/model_registry.json', 'r') as f:
    registry = json.load(f)

print("✓ All data and registry loaded")
print(f"\nRegistered Models:")
for task, info in registry.items():
    print(f"  {task}: {info['models']}")

## Cell 2: Evaluate Severity Models

In [ ]:
print("="*60)
print("SEVERITY MODEL EVALUATION")
print("="*60)

X_test = severity_data['X_test']
y_test = severity_data['y_test']

severity_models_dir = f'{PROJECT_DIR}/models/severity'
severity_results = {}

for model_name in registry['severity']['models']:
    model_path = f"{severity_models_dir}/{model_name.lower().replace(' ', '_')}.joblib"
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        severity_results[model_name] = {'Accuracy': acc, 'F1_Weighted': f1, 'F1_Macro': f1_macro}
        print(f"{model_name}: Acc={acc:.4f}, F1={f1:.4f}")

severity_df = pd.DataFrame(severity_results).T
print("\nSeverity Model Comparison:")
print(severity_df.round(4))

## Cell 3: Evaluate ETA Models

In [ ]:
print("\n="*60)
print("ETA MODEL EVALUATION")
print("="*60)

X_test = eta_data['X_test']
y_test = eta_data['y_test']

eta_models_dir = f'{PROJECT_DIR}/models/eta'
eta_results = {}

for model_name in registry['eta']['models']:
    model_path = f"{eta_models_dir}/{model_name.lower().replace(' ', '_')}.joblib"
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(np.mean((y_test - y_pred)**2))
        r2 = r2_score(y_test, y_pred)
        eta_results[model_name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
        print(f"{model_name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R2={r2:.4f}")

eta_df = pd.DataFrame(eta_results).T
print("\nETA Model Comparison:")
print(eta_df.round(4))

## Cell 4: Evaluate Bed Models

In [ ]:
print("\n="*60)
print("BED AVAILABILITY MODEL EVALUATION")
print("="*60)

X_test = bed_data['X_test']
y_test = bed_data['y_test']

bed_models_dir = f'{PROJECT_DIR}/models/bed_availability'
bed_results = {}

for model_name in registry['bed']['models']:
    model_path = f"{bed_models_dir}/{model_name.lower().replace(' ', '_')}.joblib"
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        y_pred = np.round(model.predict(X_test)).astype(int)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(np.mean((y_test - y_pred)**2))
        r2 = r2_score(y_test, y_pred)
        bed_results[model_name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
        print(f"{model_name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R2={r2:.4f}")

bed_df = pd.DataFrame(bed_results).T
print("\nBed Model Comparison:")
print(bed_df.round(4))

## Cell 5: Create Comparison Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Severity - Accuracy (0–1 scores shown as %; avoid fixed xlim that clips bars)
acc_pct = severity_df['Accuracy'] * 100
acc_pct.plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Severity Models - Accuracy')
axes[0,0].set_xlabel('Accuracy (%)')
rng = float(acc_pct.max() - acc_pct.min()) or 1.0
x_pad = max(2.0, rng * 0.08)
axes[0,0].set_xlim(max(0.0, acc_pct.min() - x_pad), min(100.0, acc_pct.max() + x_pad))
for i, v in enumerate(acc_pct):
    axes[0,0].text(v + x_pad * 0.12, i, f'{v:.2f}%', va='center')

# ETA - MAE
eta_df['MAE'].plot(kind='barh', ax=axes[0,1], color='coral')
axes[0,1].set_title('ETA Models - MAE (lower is better)')
for i, v in enumerate(eta_df['MAE']):
    axes[0,1].text(v + 0.1, i, f'{v:.2f}', va='center')

# Bed - MAE
bed_df['MAE'].plot(kind='barh', ax=axes[1,0], color='seagreen')
axes[1,0].set_title('Bed Models - MAE (lower is better)')
for i, v in enumerate(bed_df['MAE']):
    axes[1,0].text(v + 0.05, i, f'{v:.2f}', va='center')

# R2 Comparison (ETA and Bed)
x = np.arange(len(eta_df))
width = 0.35
axes[1,1].bar(x - width/2, eta_df['R2'], width, label='ETA', color='coral')
axes[1,1].bar(x + width/2, bed_df['R2'], width, label='Bed', color='seagreen')
axes[1,1].set_title('R² Score Comparison')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(eta_df.index, rotation=45, ha='right')
axes[1,1].legend()
axes[1,1].set_ylim(0, 1)

plt.suptitle('Model Performance Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/model_comparison_dashboard.png', dpi=150)
plt.show()
print(f"\n✓ Dashboard saved to {PROJECT_DIR}/results/model_comparison_dashboard.png")

## Cell 6: Save Evaluation Summary

In [ ]:
evaluation_summary = {
    'severity_models': severity_results,
    'eta_models': eta_results,
    'bed_models': bed_results,
    'best_models': {
        'severity': 'Stacking',
        'eta': 'Stacking',
        'bed': 'Stacking'
    }
}

results_dir = f'{PROJECT_DIR}/results'
os.makedirs(results_dir, exist_ok=True)

with open(f'{results_dir}/evaluation_summary.json', 'w') as f:
    json.dump(evaluation_summary, f, indent=2)

print("="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"\nSummary saved to {results_dir}/evaluation_summary.json")

print("\nBest Model Performance:")
print(f"  Severity: {severity_results.get('Stacking', {}).get('Accuracy', 0):.4f} accuracy")
print(f"  ETA: {eta_results.get('Stacking', {}).get('MAE', 0):.3f} MAE")
print(f"  Bed: {bed_results.get('Stacking', {}).get('MAE', 0):.3f} MAE")